In [1]:
import os
import shutil

# 1. Define Paths
WHEEL_DIR = "/kaggle/working/wheels_fast"
# This is the path to your uploaded dataset wheel
SOURCE_WHEEL = "/kaggle/input/datasets/pjleek/onnx-runtime126-311/onnxruntime-1.26.0-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl"

# 2. Reset and Create Directory
if os.path.exists(WHEEL_DIR):
    shutil.rmtree(WHEEL_DIR)
os.makedirs(WHEEL_DIR, exist_ok=True)

# 3. Move the "Tool" into the workspace
print(f"Moving ONNX Runtime 1.26.0 from Dataset to {WHEEL_DIR}...")

if os.path.exists(SOURCE_WHEEL):
    # Copy the file to your destination folder
    dest_path = os.path.join(WHEEL_DIR, os.path.basename(SOURCE_WHEEL))
    shutil.copy(SOURCE_WHEEL, dest_path)
else:
    print(f"❌ ERROR: Source wheel not found at {SOURCE_WHEEL}")
    print("Please check if the Dataset is correctly attached to the notebook.")

# 4. Final Audit
print("\n--- Final Directory Audit ---")
downloaded_files = os.listdir(WHEEL_DIR)
if any("onnxruntime" in f for f in downloaded_files):
    print(f"✅ onnxruntime 1.26.0 successfully staged: {downloaded_files[0]}")
else:
    print("❌ ERROR: onnxruntime is MISSING.")

Moving ONNX Runtime 1.26.0 from Dataset to /kaggle/working/wheels_fast...

--- Final Directory Audit ---
✅ onnxruntime 1.26.0 successfully staged: onnxruntime-1.26.0-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl


In [2]:
%%writefile main.py
import math
import numpy as np
import os
import sys
import zipfile
import glob
from collections import defaultdict, namedtuple

# ... (Keep existing setup and import logic: AGENT_DIR, LIB_DIR, ONNX loading) ...
#[Assumed unchanged for brevity: Imports, Path setup, ONNX Session loading]

CENTER_X, CENTER_Y = 50.0, 50.0
SUN_R, SUN_SAFETY = 10.0, 0.5
MAX_SPEED = 6.0
MIN_NN_SCORE = 0.20 

Planet = namedtuple("Planet",["id", "owner", "x", "y", "radius", "ships", "production"])
TRAPS =[(15, 83), (5, 67), (9, 76), (75, 75), (89, 61), (6, 28), (90, 68), (95, 69), (56, 9), (36, 18)]

def get_danger_heat(tx, ty):
    min_d = min(math.hypot(tx - t[0], ty - t[1]) for t in TRAPS)
    return max(0.0, 1.0 - (min_d / 15.0))

def fleet_speed(ships):
    if ships <= 1: return 1.0
    return 1.0 + (MAX_SPEED - 1.0) * (min(1.0, math.log(ships) / math.log(1000.0)) ** 1.5)

def point_to_segment_dist(px, py, x1, y1, x2, y2):
    dx, dy = x2 - x1, y2 - y1
    l2 = dx*dx + dy*dy
    if l2 == 0: return math.hypot(px - x1, py - y1)
    t = max(0.0, min(1.0, ((px - x1)*dx + (py - y1)*dy) / l2))
    return math.hypot(px - (x1 + t*dx), py - (y1 + t*dy))

def get_target_pos(tgt, eta, ang_vel, comets, comet_ids):
    """Calculates target position using floating point ETA for accuracy."""
    if tgt.id in comet_ids:
        # Comet path logic remains index-based (integer)
        for c in comets:
            if tgt.id in c.get("planet_ids",[]):
                idx = c["planet_ids"].index(tgt.id)
                f_idx = c.get("path_index", 0) + int(eta)
                if idx < len(c["paths"]) and 0 <= f_idx < len(c["paths"][idx]): 
                    return c["paths"][idx][f_idx]
        return (tgt.x, tgt.y)
    
    # Accurate rotation using float eta
    if ang_vel == 0.0: return (tgt.x, tgt.y)
    r = math.hypot(tgt.x - CENTER_X, tgt.y - CENTER_Y)
    ang = math.atan2(tgt.y - CENTER_Y, tgt.x - CENTER_X) + (ang_vel * eta)
    return (CENTER_X + r * math.cos(ang), CENTER_Y + r * math.sin(ang))

def plan_flight(src, tgt, ships, ang_vel, comets, comet_ids):
    """Refined iteration loop to converge on precise target interception."""
    speed = fleet_speed(ships)
    # Initial estimate
    tx, ty = tgt.x, tgt.y
    eta = 0.0
    
    # Fixed-point iteration to stabilize ETA and position
    for _ in range(6): 
        dist = math.hypot(tx - src.x, ty - src.y)
        flight_dist = max(0.0, dist - src.radius - tgt.radius)
        eta = flight_dist / speed
        tx, ty = get_target_pos(tgt, eta, ang_vel, comets, comet_ids)

    angle = math.atan2(ty - src.y, tx - src.x)
    
    # Collision check
    sx = src.x + math.cos(angle) * (src.radius + 0.1)
    sy = src.y + math.sin(angle) * (src.radius + 0.1)
    ex = sx + math.cos(angle) * (eta * speed)
    ey = sy + math.sin(angle) * (eta * speed)
    
    if point_to_segment_dist(CENTER_X, CENTER_Y, sx, sy, ex, ey) <= SUN_R + SUN_SAFETY: 
        return None, 999, tx, ty, speed
        
    return angle, eta, tx, ty, speed

def agent(obs, config=None):
    # ... [Keep status logging logic same] ...
    
    # [Keep obs extraction logic same]
    # ...
    
    # ... [Keep threat calculation same] ...
    
    for src in my_planets:
        # ... [Budget/Threat check same] ...
        
        for tgt in planets:
            if tgt.id == src.id: continue
            is_defensive = (tgt.owner == player)
            
            # Use floating point math for cost estimation
            angle, eta, tx, ty, speed = plan_flight(src, tgt, avail, ang_vel, comets, comet_ids)
            if angle is None: continue
            
            # Logic: Calculate ships needed based on converged ETA
            if not is_defensive:
                enemy_future = tgt.ships + (tgt.production * eta if tgt.owner != -1 else 0)
                cost = int(enemy_future * 1.1) + 1
                if cost > avail: continue
            else:
                # Defensive logic
                cost = max(1, int(threats[tgt.id] * 1.05) - tgt.ships)
                cost = min(avail, cost)
                if cost < 1: continue

            # ...[Keep NN scoring and move selection same] ...
            
    return moves

Writing main.py


In [3]:
import tarfile
import shutil
import os

working_dir = "/kaggle/working"
submission_path = os.path.join(working_dir, "submission.tar.gz")

# Hunt for the ONNX files across Kaggle's unpredictable mount paths
dataset_search_paths =[
    "/kaggle/input/datasets/pjleek/euro-step-v4",
    "/kaggle/input/euro-step-v4",
    "/kaggle/input/pjleek/euro-step-v4",
    "/kaggle/working" # In case it was generated in the same notebook session
]

def find_file(filename):
    for path in dataset_search_paths:
        full_path = os.path.join(path, filename)
        if os.path.exists(full_path):
            return full_path
    return None

print("🔍 Locating ONNX files...")
onnx_src = find_file("orbit_model.onnx")
onnx_data_src = find_file("orbit_model.onnx.data")

if not onnx_src:
    print("❌ CRITICAL ERROR: Could not find orbit_model.onnx! Is your dataset attached to the notebook?")
else:
    # Copy to working directory
    if onnx_src != os.path.join(working_dir, "orbit_model.onnx"):
        shutil.copy(onnx_src, os.path.join(working_dir, "orbit_model.onnx"))
    print(f"✅ Found and staged: {onnx_src}")

if onnx_data_src:
    if onnx_data_src != os.path.join(working_dir, "orbit_model.onnx.data"):
        shutil.copy(onnx_data_src, os.path.join(working_dir, "orbit_model.onnx.data"))
    print(f"✅ Found and staged: {onnx_data_src}")
else:
    print("⚠️ Note: orbit_model.onnx.data not found. (If your model is small, it doesn't need this, which is fine).")

print("\n📦 Creating submission.tar.gz...")
try:
    with tarfile.open(submission_path, "w:gz") as tar:
        # 1. Add main.py
        if os.path.exists(os.path.join(working_dir, "main.py")):
            tar.add(os.path.join(working_dir, "main.py"), arcname="main.py")
            print("   ➕ Added main.py")
        else:
            print("   ❌ ERROR: main.py is missing!")
            
        # 2. Add ONNX Brain & Data
        if os.path.exists(os.path.join(working_dir, "orbit_model.onnx")):
            tar.add(os.path.join(working_dir, "orbit_model.onnx"), arcname="orbit_model.onnx")
            print("   ➕ Added orbit_model.onnx")
        else:
            print("   ❌ FAILED to add orbit_model.onnx")
            
        if os.path.exists(os.path.join(working_dir, "orbit_model.onnx.data")):
            tar.add(os.path.join(working_dir, "orbit_model.onnx.data"), arcname="orbit_model.onnx.data")
            print("   ➕ Added orbit_model.onnx.data")
            
        # 3. Add Wheels Folder
        wheels_dest = os.path.join(working_dir, "wheels_fast")
        if os.path.exists(wheels_dest):
            tar.add(wheels_dest, arcname="wheels_fast")
            print("   ➕ Added wheels_fast")
        else:
            print("   ❌ FAILED to add wheels_fast folder! Did you run the wheel downloader?")
            
    print(f"\n🎉 SUCCESS! File ready for leaderboard: {submission_path}")
        
except Exception as e:
    print(f"\n❌ FAILED to create tarball: {e}")

🔍 Locating ONNX files...
✅ Found and staged: /kaggle/input/datasets/pjleek/euro-step-v4/orbit_model.onnx
✅ Found and staged: /kaggle/input/datasets/pjleek/euro-step-v4/orbit_model.onnx.data

📦 Creating submission.tar.gz...
   ➕ Added main.py
   ➕ Added orbit_model.onnx
   ➕ Added orbit_model.onnx.data
   ➕ Added wheels_fast

🎉 SUCCESS! File ready for leaderboard: /kaggle/working/submission.tar.gz


In [4]:
import tarfile

# Define the path to your submission archive
submission_path = "/kaggle/working/submission.tar.gz"

try:
    with tarfile.open(submission_path, "r:gz") as tar:
        print(f"📦 Contents of {submission_path}:")
        for name in tar.getnames():
            print(f"  - {name}")
except Exception as e:
    print(f"❌ Error opening archive: {e}")

📦 Contents of /kaggle/working/submission.tar.gz:
  - main.py
  - orbit_model.onnx
  - orbit_model.onnx.data
  - wheels_fast
  - wheels_fast/onnxruntime-1.26.0-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl
